# Gemma 4 31B FBSFM fine-tune - v1 (cleaned)

Cleaned copy of `gemma_training_colab.ipynb` for the adapter you **already trained** (`gemma_fbsfm_lora`, 3319/3319 steps, final loss ~1.07). Your original notebook is untouched.

Changes:
- Removed the broken / duplicate cells (old 16-24): `prompt_template` was undefined, one cell loaded the adapter without `text_only=True` (PEFT reported *missing adapter keys* - the LoRA was **not applied**), and the JSON parse failed on Gemma 4's `<|channel>thought\n<channel|>` prefix.
- Training cell is kept for reference but switched off (`RUN_TRAINING = False`); it now resumes only if a checkpoint exists.
- Validation and extraction call two scripts in `gemma_ft/`:
  - `validate_gemma_lora.py` - prompts the model in its **training** format (v1), scores JSON validity, truncation, schema, grounding in the page text and P/R/F1 vs ground truth.
  - `colab_other_extract_v4.py` - loads the 31B base via Unsloth (v3 loaded `gemma-4-E4B-it`, which cannot take this adapter), uses the v1 prompt format, salvages truncated output, and its resume check now finds finished pages.

The prompt mismatch between training and extraction is fixed properly in **`gemma_training_v2_promptfix.ipynb`** (retraining).

## 1. Install

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ROOT        = "/content/drive/MyDrive/gemma_ft"
DATASET     = f"{ROOT}/gemma_training_dataset.jsonl.gz"
ADAPTER_DIR = f"{ROOT}/gemma_fbsfm_lora"
OUTPUT_DIR  = f"{ROOT}/outputs"

## 3. Training (already done - leave `RUN_TRAINING = False`)

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    import glob, torch
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from datasets import load_dataset
    from trl import SFTTrainer
    from transformers import TrainingArguments

    max_seq_length = 5120
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/gemma-4-31B-it-unsloth-bnb-4bit",
        max_seq_length = max_seq_length, dtype = None, load_in_4bit = True, text_only = True)
    model = FastLanguageModel.get_peft_model(
        model, r = 32,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 32, lora_dropout = 0, bias = "none",
        use_gradient_checkpointing = "unsloth", random_state = 3407)

    dataset = load_dataset("json", data_files=DATASET, split="train")
    def format_prompts(examples):
        texts = []
        for i, o in zip(examples["raw_ocr_text"], examples["extracted_graph"]):
            messages = [
                {"role": "user", "content": f"Below is an OCR page from a biology textbook. Extract the knowledge graph in JSON format.\n\n### Page Text:\n{i}"},
                {"role": "model", "content": f"### Extracted JSON:\n{o}"},
            ]
            texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
        return {"text": texts}
    dataset = dataset.map(format_prompts, batched=True)

    trainer = SFTTrainer(
        model = model, tokenizer = tokenizer, train_dataset = dataset,
        dataset_text_field = "text", max_seq_length = max_seq_length, dataset_num_proc = 1, packing = False,
        args = TrainingArguments(
            per_device_train_batch_size = 8, gradient_accumulation_steps = 2, warmup_steps = 5,
            num_train_epochs = 1, learning_rate = 2e-4,
            fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
            logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.01, lr_scheduler_type = "linear",
            seed = 3407, output_dir = OUTPUT_DIR, save_strategy = "steps", save_steps = 50, save_total_limit = 2),
    )
    ckpts = glob.glob(f"{OUTPUT_DIR}/checkpoint-*")
    trainer.train(resume_from_checkpoint = bool(ckpts))   # True with no checkpoint raises an error
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
else:
    print("Training skipped - adapter already trained and saved in", ADAPTER_DIR)

## 4. Checkpoints

In [ ]:
import os, glob, datetime
for cp in sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda x: int(x.split("-")[-1])):
    size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(cp) for f in fs) / 2**20
    when = datetime.datetime.fromtimestamp(os.path.getmtime(cp)).strftime("%Y-%m-%d %H:%M")
    print(f"{os.path.basename(cp):18s} {when}  {size:,.0f} MB")
print("adapter:", os.listdir(ADAPTER_DIR))

## 5. Validate

Run in a session where no model is loaded in this notebook (fresh session, or **Runtime -> Restart session** and re-run cell 2). The script uses the v1 training prompt format automatically.

In [ ]:
# pages from the training file (seen in training): did the model learn the format?
!python {ROOT}/validate_gemma_lora.py --adapter {ADAPTER_DIR} --data train --n 20 --report {ROOT}/validation_v1_train.json

# unseen pages from other_resources_in: does it generalise / stay grounded?
!python {ROOT}/validate_gemma_lora.py --adapter {ADAPTER_DIR} --data chunk --n 20 --report {ROOT}/validation_v1_chunk.json

## 6. Batch extraction (v4 script)

In [ ]:
!python -u {ROOT}/colab_other_extract_v4.py \
    --input {ROOT}/other_resources_in \
    --adapter {ADAPTER_DIR} \
    --out {ROOT}/other_resources_out \
    --batch 4 --max-new 8192